# 🚨 Step 4 — Score Transactions & Generate Fraud Alerts (Gold)

Apply the trained model to all transactions.
Flag suspicious ones with a fraud probability score.

In [ ]:
import mlflow.sklearn
import pandas as pd

# Load the latest fraud model from MLflow
model_uri = 'runs:/{run_id}/fraud_model'  # Replace run_id from previous notebook output
# For demo: reload model from training session
# model = mlflow.sklearn.load_model(model_uri)

# Re-train inline for demo continuity
from sklearn.ensemble import GradientBoostingClassifier
df_all = spark.table('silver_fraud_features').toPandas()
features = ['Amount','TimeSinceLastTxnMins','NumTxnLast24h','AvgTxnAmount30d',
            'location_jump','high_velocity','amount_spike','fast_repeat']
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(df_all[features], df_all['IsFraud'].astype(int))
print('✅ Model ready for scoring')

In [ ]:
# Score all transactions
df_score = df_all.copy()
df_score['FraudProbability'] = model.predict_proba(df_all[features])[:, 1]
df_score['FraudAlert'] = df_score['FraudProbability'] > 0.5

# Show flagged transactions
alerts = df_score[df_score['FraudAlert']][['TransactionID','AccountID','Amount','Location','FraudProbability']]
alerts = alerts.sort_values('FraudProbability', ascending=False)

print(f'🚨 FRAUD ALERTS: {len(alerts)} suspicious transactions detected!')
print()
print(alerts.to_string(index=False))

In [ ]:
# Save scored results to Gold layer
df_gold = spark.createDataFrame(df_score[['TransactionID','AccountID','Amount',
    'TransactionDate','Location','FraudProbability','FraudAlert','IsFraud']])

df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_fraud_alerts')
print('\n✅ Gold fraud alerts table saved — ready for Power BI dashboard!')

In [ ]:
%%sql
-- Final view: All high-risk fraud alerts
SELECT TransactionID, AccountID, Amount, Location,
       ROUND(FraudProbability * 100, 1) AS FraudScore_Pct,
       CASE WHEN FraudAlert = true THEN '🚨 ALERT' ELSE '✅ OK' END AS Status
FROM gold_fraud_alerts
ORDER BY FraudProbability DESC